# Pipeline explorer

Run one document through the pipeline and watch each stage. Change the settings
in the next cell, then **Run All**.

Every stage shows what it actually produced as a table, followed by a note on what
that stage did and what tends to go wrong in it.


## Settings

Edit these lines. Everything below follows from them.

| setting | options |
|---|---|
| `SAMPLE` | `1`–`10` for a document from the evaluation set, or `None` to use `PDF_PATH` |
| `PDF_PATH` | path to any PDF — used only when `SAMPLE = None` |
| `MODEL` | `'llama3.1:8b'`, `'gemma4:e4b'`, `'llama3.3:70b'` |
| `EXTRACTOR` | `'pdfplumber'`, `'docling'`, `'lighton'` |
| `RERUN` | `False` reuses saved output; `True` calls the model again |
| `SHOW_MERGE` | `True` re-extracts without line merging to show what it changed (pdfplumber only) |

> With `RERUN = False` this opens instantly if the combination has been run
> before. Set it to `True` after changing the prompt, or the notebook will show
> you output from the old one.


In [1]:
SAMPLE     = 1                  # 1-10, or None to use PDF_PATH
PDF_PATH   = 'data/input/pdfs/sample1.pdf'
MODEL      = 'llama3.1:8b'
EXTRACTOR  = 'pdfplumber'
RERUN      = False
SHOW_MERGE = True


In [2]:
import json
import os
import time
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import pandas as pd
from IPython.display import display

from dmpbridge.core import paths as P
from dmpbridge.core.pipeline import run_and_save
from dmpbridge.strategies.wholedoc import WholeDocStrategy

pd.set_option('display.max_colwidth', 78)
pd.set_option('display.width', 180)

TAG = P.make_tag(MODEL, EXTRACTOR)

if SAMPLE is not None:
    pdf   = Path(f'data/input/pdfs/sample{SAMPLE}.pdf')
    stem  = f'sample{SAMPLE}'
    paths = {2: P.labeled_path(TAG, SAMPLE), 3: P.structured_path(TAG, SAMPLE),
             4: P.final_path(TAG, SAMPLE)}
else:
    # A PDF from outside the evaluation set: keep its output out of the
    # tagged folders, which are reserved for scored runs.
    pdf   = Path(PDF_PATH)
    stem  = pdf.stem
    scratch = P.OUTPUT_ROOT / 'explorer' / TAG
    paths = {n: scratch / f'{stem}__stage{n}.json' for n in (2, 3, 4)}

if not pdf.exists():
    raise FileNotFoundError(f'No PDF at {pdf}')

display(pd.DataFrame([['document', str(pdf)], ['model', MODEL],
                      ['extractor', EXTRACTOR], ['tag', TAG]],
                     columns=['setting', 'value']).set_index('setting'))

missing = [n for n, p in paths.items() if not p.exists()]
if RERUN or missing:
    why = 'RERUN = True' if RERUN else f'stage(s) {missing} not yet produced'
    print(f'Running the pipeline ({why}) — this calls {MODEL}.')
    t0 = time.time()
    strategy = WholeDocStrategy(model=MODEL, extractor=EXTRACTOR,
                                cache_dir=P.EXTRACTED_DIR / EXTRACTOR)
    run_and_save(strategy, pdf, paths[2], struct_path=paths[3], final_path=paths[4])
    print(f'done in {time.time() - t0:.1f} s')
else:
    print('Using saved output. Set RERUN = True to call the model again.')


,value
setting,
document,data\input\pdfs\sample1.pdf
model,llama3.1:8b
extractor,pdfplumber
tag,llama3.1-8b_pdfplumber_whole_doc


Using saved output. Set RERUN = True to call the model again.


## Stage 1 — read the PDF

The extractor turns pages into text blocks. Nothing is classified yet; this step
has no idea what a question or an answer is.


In [3]:
raw = json.loads((P.EXTRACTED_DIR / EXTRACTOR / f'{stem}.json').read_text(encoding='utf-8'))
blocks1 = raw if isinstance(raw, list) else raw.get('blocks', [])

df1 = pd.DataFrame([{
    'page':  b.get('page'),
    'bold':  b.get('bold'),
    'chars': len(b.get('text', '')),
    'text':  b.get('text', ''),
} for b in blocks1])
df1.index.name = 'block'

print(f'{len(df1)} blocks, {df1["chars"].sum()} characters')
display(df1.head(12))


44 blocks, 6117 characters


,page,bold,chars,text
block,,,,
0,1,None,32,DATA MANAGEMENT AND SHARING PLAN
1,1,None,21,Element 1: Data Type:
2,1,None,79,A. Types and amount of scientific data expected to be generated in the pro...
3,1,None,208,This secondary data analysis project will analyze deidentified data from 4...
4,1,None,222,"The studies include (i) the RISE Study, (ii) the SOL-VIDA Study, (iii) the..."
5,1,None,85,"B. Scientific data that will be preserved and shared, and the rationale fo..."
6,1,None,241,"As this is a secondary data analysis project, we will only be able to publ..."
7,1,None,18,(i) The RISE study
8,1,None,21,(ii) The iWATCH study


**What just happened.** The PDF became a flat list of text blocks, each carrying
position and font information. The model never sees the PDF itself — only this
list, so anything lost here is lost for good.

**What goes wrong here.** `pdfplumber` reads the PDF text layer line by line, so one
paragraph arrives as several blocks. The next section shows that being repaired.
`docling` and `lighton` segment by paragraph natively and skip it.

This stage is cached per *extractor* and shared by every model, which is why
changing `MODEL` above does not re-extract.


### What line merging changed  ·  pdfplumber only

`merge_wrapped_lines()` runs **inside** `PdfplumberExtractor.extract()`, so what you
saw above is already merged — the raw lines are never written to disk. This
re-extracts with merging switched off so the two can be compared.

Docling and LightOnOCR never call it: this repairs a pdfplumber-specific defect,
it is not a general pipeline stage.


In [4]:
def merge_groups(raw_blocks, merged_blocks):
    """Walk both lists in order, grouping the raw lines that formed each block."""
    squash = lambda s: ''.join(s.split())
    groups, i = [], 0
    for m in merged_blocks:
        want, got, parts = squash(m.get('text', '')), '', []
        while i < len(raw_blocks) and len(got) < len(want):
            parts.append(raw_blocks[i].get('text', ''))
            got += squash(raw_blocks[i].get('text', ''))
            i += 1
        groups.append(parts)
    return groups


if EXTRACTOR != 'pdfplumber':
    print(f'{EXTRACTOR} does not use line merging — nothing to compare.')
elif not SHOW_MERGE:
    print('SHOW_MERGE is False. Set it to True to re-extract without merging.')
else:
    from dmpbridge.extractors import get_extractor

    unmerged = get_extractor('pdfplumber', merge_lines=False).extract(pdf)
    groups   = merge_groups(unmerged, blocks1)

    display(pd.DataFrame([
        {'blocks': len(unmerged), 'characters': sum(len(b.get('text', '')) for b in unmerged)},
        {'blocks': len(blocks1),  'characters': sum(len(b.get('text', '')) for b in blocks1)},
    ], index=['raw lines', 'after merging']))

    joined = [(i, g) for i, g in enumerate(groups) if len(g) > 1]
    print(f'{len(joined)} of {len(blocks1)} blocks were built from more than one line.')

    rows = []
    for bi, parts in joined[:6]:
        for j, part in enumerate(parts):
            rows.append({'block': bi, 'line': j + 1, 'raw line': part})
        rows.append({'block': bi, 'line': '=', 'raw line': blocks1[bi].get('text', '')})
    if rows:
        display(pd.DataFrame(rows).set_index(['block', 'line']))
        print('Rows marked = are the merged result of the lines above them.')


,blocks,characters
raw lines,78,6083
after merging,44,6117


16 of 44 blocks were built from more than one line.


raw line
block line                                                                               
3     1     This secondary data analysis project will analyze deidentified data from 4...
      2     and the publicly available NHANES cohorts (wrist NHANES 2011-2014; hip/cou...
      =     This secondary data analysis project will analyze deidentified data from 4...
4     1     The studies include (i) the RISE Study, (ii) the SOL-VIDA Study, (iii) the...
      2     (v) the PHASE Study, (vi) the AusDiab Study, (vii) the ACT Study, and (vii...
      3                                                                            study.
      =     The studies include (i) the RISE Study, (ii) the SOL-VIDA Study, (iii) the...
6     1     As this is a secondary data analysis project, we will only be able to publ...
      2     Repository sitting behavior metrics or analysis that were created in this ...
      3                                                             based study datasets:
      =     As this is a secondary data analysis project, we will only be able to publ...
10    1     Sufficient data from those datasets will be preserved to enable sharing to...
      2     findings described in the Aims with those datasets only. Please see Elemen...
      =     Sufficient data from those datasets will be preserved to enable sharing to...
13    1     In addition to the data described above, code and models will be included ...
      2     project’s GitHub website, hosted as part of the UCSD Advance Data Analytic...
      3                                                                        Postures”.
      =     In addition to the data described above, code and models will be included ...
15    1     Data will be analyzed with custom code by our statistical and computer sci...
      2     processed and analyzed using ActiLife software which requires a paid licen...
      3     will also be shared in the data repository and on our project’s GitHub web...
      4                     Advance Data Analytics Lab GitHub and titled “Deep Postures”.
      =     Data will be analyzed with custom code by our statistical and computer sci...

Rows marked = are the merged result of the lines above them.


**Why this matters.** A wrapped answer split across six lines is six chances for the
model to label a sentence fragment as something it is not. Merging first was worth
**+28.4 points of F1** when it was added.

The decisive signal is the **right margin**, not punctuation: a line that stops well
short of the margin has ended a paragraph, while one that runs to the margin has
wrapped. Using punctuation alone wrongly merged headings into the text below them.

To turn it off in code: `get_extractor('pdfplumber', merge_lines=False)`.


## Stage 2 — label every block

The whole document goes to the model in **one call**, and it returns a label and a
confidence for each block.


In [5]:
raw2 = json.loads(paths[2].read_text(encoding='utf-8'))
blocks2 = raw2 if isinstance(raw2, list) else raw2.get('blocks', [])

df2 = pd.DataFrame([{
    'label':      b.get('label'),
    'confidence': b.get('confidence'),
    'text':       b.get('text', ''),
} for b in blocks2])
df2.index.name = 'block'

display(df2['label'].value_counts().rename('blocks').to_frame())
display(df2.head(12))


,blocks
label,
answer.text,27
section.title,14
question.text,2
title,1


,label,confidence,text
block,,,
0,title,1.0,DATA MANAGEMENT AND SHARING PLAN
1,section.title,1.0,Element 1: Data Type:
2,question.text,1.0,A. Types and amount of scientific data expected to be generated in the pro...
3,answer.text,1.0,This secondary data analysis project will analyze deidentified data from 4...
4,answer.text,1.0,"The studies include (i) the RISE Study, (ii) the SOL-VIDA Study, (iii) the..."
5,section.title,1.0,"B. Scientific data that will be preserved and shared, and the rationale fo..."
6,answer.text,1.0,"As this is a secondary data analysis project, we will only be able to publ..."
7,answer.text,1.0,(i) The RISE study
8,answer.text,1.0,(ii) The iWATCH study


**What just happened.** Each block now has one of five labels — `title`,
`section.title`, `section.description`, `question.text`, `answer.text` — plus the
model's own confidence. The definitions come from
[`dmpbridge/prompts/system.py`](../dmpbridge/prompts/system.py).

**What goes wrong here.** Almost every pipeline error starts in this stage, and the
hardest boundary is `question.text` versus `section.title`: a lettered sub-item
(`A.`, `B.`, `C.`) under a heading is a *question*, but it looks like a heading.
Compare the counts above against what the document really contains — if a document
with nine real questions shows only two, they have already been lost here.

Confidence is the model's own claim and is close to `1.00` almost everywhere,
including on wrong labels. Do not read it as reliability.


## Stage 3 — build the structure

The flat list becomes a nested document: sections, each holding questions, each
holding an answer. No model call — this is deterministic, driven entirely by the
labels from stage 2.


In [6]:
def unpack(path):
    """(title, sections) from the DMP-tool narrative schema."""
    tpl = json.loads(path.read_text(encoding='utf-8')).get('narrative', {}).get('template', {})
    return tpl.get('title', ''), tpl.get('section', [])


def to_frame(sections):
    """One row per question, carrying its section."""
    rows = []
    for si, s in enumerate(sections, 1):
        qs = s.get('question', [])
        if not qs:
            rows.append({'section': si, 'section title': s.get('title', ''),
                         'question': '', 'answer': ''})
        for q in qs:
            ans = (q.get('answer') or {}).get('json', {}).get('answer', '')
            rows.append({'section': si, 'section title': s.get('title', ''),
                         'question': str(q.get('text', '')), 'answer': str(ans)})
    return pd.DataFrame(rows)


title3, sections3 = unpack(paths[3])
df3 = to_frame(sections3)
print(f'title: {title3!r}')
print(f'{len(sections3)} sections, {df3["question"].ne("").sum()} questions with text')
display(df3.head(12))


title: 'DATA MANAGEMENT AND SHARING PLAN'
14 sections, 2 questions with text


,section,section title,question,answer
0,1,Element 1: Data Type:,A. Types and amount of scientific data expected to be generated in the pro...,This secondary data analysis project will analyze deidentified data from 4...
1,2,"B. Scientific data that will be preserved and shared, and the rationale fo...",,"As this is a secondary data analysis project, we will only be able to publ..."
2,3,"C. Metadata, other relevant data, and associated documentation:",,"In addition to the data described above, code and models will be included ..."
3,4,"Element 2: Related Tools, Software and/or Code:",,Data will be analyzed with custom code by our statistical and computer sci...
4,5,Element 3: Standards:,The following data will be created as a result of this project:,Objective sedentary behavior metrics – no existing standards
5,6,"Element 4: Data Preservation, Access, and Associated Timelines:",,
6,7,A. Repository where scientific data and metadata will be archived:,,De-identified data will be uploaded to the University of California San Di...
7,8,B. How scientific data will be findable and identifiable:,,Data will be findable via publications and on our project’s GitHub website...
8,9,C. When and how long the scientific data will be made available:,,The research community will have access to data when the main outcome manu...
9,10,"Element 5: Access, Distribution, or Reuse Considerations:",,


**What just happened.** Consecutive blocks sharing a label were merged, and the
hierarchy rebuilt: every `section.title` opens a new section, every `question.text`
opens a question inside it, and following `answer.text` becomes that question's
answer. This is the shape the DMP Tool expects.

**What goes wrong here.** Nothing is invented at this stage, but stage 2's mistakes
change shape. A question mislabelled as a heading does not merely lose one label —
it **opens a whole new section**, so a single wrong label restructures the document.
That is why the section count can be far higher than the document really has, and
why rows above may show a section title with an empty question beneath it.


## Stage 4 — apply the annotation rules

A deterministic pass from `data/input/Rules.xlsx`: where a question has no text, it
is filled from the section heading, then the section description, then the document
title — whichever exists first.


In [7]:
title4, sections4 = unpack(paths[4])
df4 = to_frame(sections4)

display(pd.DataFrame([
    {'sections': len(sections3), 'questions with text': int(df3['question'].ne('').sum())},
    {'sections': len(sections4), 'questions with text': int(df4['question'].ne('').sum())},
], index=['stage 3', 'stage 4 (rules applied)']))

changed = pd.DataFrame({
    'was': df3['question'],
    'now': df4['question'],
    'filled from': df4['section title'],
})
changed = changed[changed['was'] != changed['now']]
print(f'{len(changed)} question(s) filled in by the rules')
if len(changed):
    display(changed.head(10))


,sections,questions with text
stage 3,14,2
stage 4 (rules applied),14,12


10 question(s) filled in by the rules


,was,now,filled from
1,,"B. Scientific data that will be preserved and shared, and the rationale fo...","B. Scientific data that will be preserved and shared, and the rationale fo..."
2,,"C. Metadata, other relevant data, and associated documentation:","C. Metadata, other relevant data, and associated documentation:"
3,,"Element 2: Related Tools, Software and/or Code:","Element 2: Related Tools, Software and/or Code:"
6,,A. Repository where scientific data and metadata will be archived:,A. Repository where scientific data and metadata will be archived:
7,,B. How scientific data will be findable and identifiable:,B. How scientific data will be findable and identifiable:
8,,C. When and how long the scientific data will be made available:,C. When and how long the scientific data will be made available:
10,,"A. Factors affecting subsequent access, distribution, or reuse of scientif...","A. Factors affecting subsequent access, distribution, or reuse of scientif..."
11,,B. Whether access to scientific data will be controlled:,B. Whether access to scientific data will be controlled:
12,,"Protections for privacy, rights, and confidentiality of human research par...","Protections for privacy, rights, and confidentiality of human research par..."
13,,Element 6: Oversight of Data Management and Sharing:,Element 6: Oversight of Data Management and Sharing:


**What just happened.** Stage 3 is kept unconverted and stage 4 written beside it,
so the two can be compared — which is what the table above does.

**What goes wrong here.** The rules only fill questions that are *empty*. They cannot
repair a question mislabelled as a heading in stage 2, because that question is not
blank — it is missing entirely, and its text is sitting in a section title instead.

Note what that means when you read the two tables together: a question the model
turned into a heading leaves an empty question inside that new section, which the
rules then fill *from the heading*. The text arrives in roughly the right place by a
completely different route than intended.


## Scoring — the two paths

The pipeline is scored twice, before and after the rules:

| | what is scored | against |
|---|---|---|
| **Path A** | stage 3, as the model produced it | the original annotation |
| **Path B** | stage 4, after the rules ran | the revised annotation |

Path A measures the model alone. Path B measures the model *plus* the deterministic
rules, which is what the pipeline actually delivers.

The two use **different reference versions**, so the gap between them is the
contribution of the rules — not a second opinion on the same quantity. Supports can
differ too, which is why the counts below may not match.

Only the 10 evaluation samples have annotations; any other PDF cannot be scored.


In [8]:
def score_path(pred_path, gold_path):
    """Match one stage's output against one annotation version."""
    records, no_gold = _match_structured(pred_path, extract_gold(gold_path))
    return micro_prf1(_confusion_from_match(records, no_gold)), records, no_gold


def mistakes(records, no_gold):
    """Every wrong label and every block with no counterpart, as a frame."""
    return pd.DataFrame(
        [{'annotation says': r['gold_label'], 'model said': r['pred_label'],
          'text': r['pred_text']}
         for r in records if r['pred_label'] and r['pred_label'] != r['gold_label']]
        + [{'annotation says': 'not in annotation', 'model said': lab, 'text': t}
           for t, lab in no_gold])


if SAMPLE is None:
    print('No reference annotation for a user-supplied PDF — nothing to score.')
    print('The output above is the pipeline result; correctness is for you to judge.')
else:
    from dmpbridge.evaluation.annotation_rules import resolve_new_gt_path
    from dmpbridge.evaluation.evaluate import (
        _confusion_from_match, _match_structured, extract_gold, micro_prf1,
        resolve_old_gt_path,
    )

    a, rec_a, ng_a = score_path(paths[3], resolve_old_gt_path(SAMPLE))
    b, rec_b, ng_b = score_path(paths[4], resolve_new_gt_path(SAMPLE))

    both = pd.DataFrame(
        [[a['tp'], b['tp']], [a['fp'], b['fp']], [a['fn'], b['fn']],
         [a['precision'], b['precision']], [a['recall'], b['recall']],
         [a['f1'], b['f1']]],
        index=['TP', 'FP', 'FN', 'precision', 'recall', 'f1-score'],
        columns=['Path A  (stage 3)', 'Path B  (stage 4)'])
    display(both.round(3))

    d = b['f1'] - a['f1']
    verdict = 'unchanged' if abs(d) < 0.001 else ('better' if d > 0 else 'worse')
    print(f'The rules made this document {verdict} ({d:+.3f} f1).')
    print('Note the two paths use different annotation versions, so a small '
          'difference\nmay be the reference changing rather than the rules helping.')

    err_a, err_b = mistakes(rec_a, ng_a), mistakes(rec_b, ng_b)

    summary = pd.concat([
        (err_a['annotation says'] + '  ->  ' + err_a['model said'])
        .value_counts().rename('Path A'),
        (err_b['annotation says'] + '  ->  ' + err_b['model said'])
        .value_counts().rename('Path B'),
    ], axis=1).fillna(0).astype(int)
    summary['change'] = summary['Path B'] - summary['Path A']
    print(f'\nMistakes by kind — {len(err_a)} in Path A, {len(err_b)} in Path B:')
    display(summary.sort_values('Path A', ascending=False))


,Path A (stage 3),Path B (stage 4)
TP,19.000,19.000
FP,10.000,10.000
FN,9.000,9.000
precision,0.655,0.655
recall,0.679,0.679
f1-score,0.667,0.667


The rules made this document unchanged (+0.000 f1).
Note the two paths use different annotation versions, so a small difference
may be the reference changing rather than the rules helping.

Mistakes by kind — 10 in Path A, 10 in Path B:


,Path A,Path B,change
question.text -> section.title,8,8,0
answer.text -> question.text,1,1,0
not in annotation -> answer.text,1,1,0


**Reading this.** A block is matched to the annotation by shared words, then judged
on its label. `f1-score` combines precision and recall and stays low unless both are
high, so a model cannot score well by labelling very little, or by labelling
everything it can think of.

**The `change` column is the one to read.** It shows which mistakes the rules
repaired and which they left alone. A rule can only fill a question that is *empty*,
so an error where the model put a question's text into a section heading stays put —
that question is not blank, it is missing.

The per-kind table matters more than the totals. If one row dominates — the same
`question.text -> section.title` repeated — that is a single systematic problem worth
fixing, not many separate accidents.


In [9]:
if SAMPLE is not None and len(err_a):
    print('Path A — every mistake on this document:')
    display(err_a)


Path A — every mistake on this document:


,annotation says,model said,text
0,question.text,section.title,"B. Scientific data that will be preserved and shared, and the rationale fo..."
1,question.text,section.title,"C. Metadata, other relevant data, and associated documentation:"
2,answer.text,question.text,The following data will be created as a result of this project:
3,question.text,section.title,A. Repository where scientific data and metadata will be archived:
4,question.text,section.title,B. How scientific data will be findable and identifiable:
5,question.text,section.title,C. When and how long the scientific data will be made available:
6,question.text,section.title,"A. Factors affecting subsequent access, distribution, or reuse of scientif..."
7,question.text,section.title,B. Whether access to scientific data will be controlled:
8,question.text,section.title,"Protections for privacy, rights, and confidentiality of human research par..."
9,not in annotation,answer.text,Objective sedentary behavior metrics – no existing standards


---

**Try next:** change `MODEL` in the settings cell and Run All. Extraction is cached,
so only the labeling re-runs, and any difference you see is the model's alone.
